In [33]:
# =============================================================================
# Fig5: City-level net inflow & local change scatter (2020→2060)
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker
import matplotlib.lines as mlines
from adjustText import adjust_text

# ── 1. Paths ──────────────────────────────────────────────────────────────────

FLOW_DIR  = Path("/Volumes/UCL/论文工作/空气污染/cross_flow/flow_results/air_scenarios")
LOCAL_DIR = Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5/city_patient_sum")
SHP_PATH  = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/city_shp/shi_en.shp")
OUTFILE   = Path("/Users/shirley/Desktop/plots_V2/Fig5_scatter_inflow_vs_local.png")
SCENARIO  = "earlypeak_NZ_CL"

CITY_NAME_MAP = {
    "Suuzhou":    "Suzhou",
    "Wulumuqi":   "Urumqi",
    "Xian":       "Xi'an",
    "Qiqihaer":   "Qiqihar",
    "Huhehaote":  "Hohhot",
    "Haerbin":    "Harbin",
}


# ── 2. Region map ─────────────────────────────────────────────────────────────

HHY = {"lon": [127.5, 98.5], "lat": [50.2, 25.0]}

def hhy_lon_at_lat(lat):
    t = (lat - HHY["lat"][1]) / (HHY["lat"][0] - HHY["lat"][1])
    return HHY["lon"][1] + t * (HHY["lon"][0] - HHY["lon"][1])

city_shp = gpd.read_file(SHP_PATH)
cx = city_shp.geometry.centroid.x
cy = city_shp.geometry.centroid.y
city_shp["region"] = np.where(cx > hhy_lon_at_lat(cy), "East", "West")
region_map = (
    city_shp[["English", "region"]]
    .rename(columns={"English": "city"})
    .drop_duplicates(subset="city")
    .set_index("city")["region"]
    .to_dict()
)

# ── 3. 数据读取函数 ───────────────────────────────────────────────────────────

def load_flow_matrix(year):
    path = FLOW_DIR / f"flow_patientnum_avg_{SCENARIO}_{year}.csv"
    df   = pd.read_csv(path, index_col=0)
    df.index   = df.index.str.strip()
    df.columns = df.columns.str.strip()
    df = df[df.index.notna()]
    df = df.loc[~df.index.isin(["total"]), ~df.columns.isin(["total"])]
    np.fill_diagonal(df.values, 0)
    return df

def load_local_patients(year):
    path = LOCAL_DIR / f"citysum_{SCENARIO}_{year}.csv"
    df   = pd.read_csv(path)
    df["city"] = df["city"].str.strip()
    return df.groupby("city")["local_patient"].sum().rename(year)

def compute_components(year):
    df      = load_flow_matrix(year)
    inflow  = df.sum(axis=0)
    outflow = df.sum(axis=1)
    net     = inflow - outflow
    local   = load_local_patients(year)
    common  = net.index.intersection(local.index)
    return net[common], local[common]

# ── 4. 计算变化量 ─────────────────────────────────────────────────────────────

net_2020, local_2020 = compute_components(2020)
net_2060, local_2060 = compute_components(2060)

common = (net_2020.index
          .intersection(net_2060.index)
          .intersection(local_2020.index)
          .intersection(local_2060.index))

df_plot = pd.DataFrame({
    "city":        common,
    "delta_net":   (net_2060   - net_2020  )[common].values,
    "delta_local": (local_2060 - local_2020)[common].values,
    "region":      pd.Series(region_map).reindex(common).values,
}).dropna()

df_plot["delta_total"] = df_plot["delta_net"] + df_plot["delta_local"]
df_plot["city_label"] = df_plot["city"].replace(CITY_NAME_MAP)
print(f"总城市数: {len(df_plot)}")
print(df_plot.groupby("region")[["delta_net","delta_local","delta_total"]].describe().round(0))

# ── 5. 绘图 ───────────────────────────────────────────────────────────────────

plt.rcParams.update({"font.family": "Arial", "font.size": 6})

fig, ax = plt.subplots(figsize=(9/2.54, 9/2.54), dpi=400)

COLORS  = {"East": "#4db3b3", "West": "#ca0020"}
MARKERS = {"East": "o",       "West": "s"}

# 点大小 = |delta_total|
size_vals = np.abs(df_plot["delta_total"].values)
size_norm = (size_vals / size_vals.max()) * 60 + 8   # 8 ~ 68

texts = []
for region in ["West", "East"]:
    sub   = df_plot[df_plot["region"] == region]
    sizes = (np.abs(sub["delta_total"].values) / size_vals.max()) * 60 + 8

    ax.scatter(
        sub["delta_net"], sub["delta_local"],
        c=COLORS[region], marker=MARKERS[region],
        s=sizes, alpha=0.6, lw=0.2, edgecolors="white",
        zorder=3
    )

    # 标注极端城市
    extremes = pd.concat([
        sub.nlargest(3,  "delta_net"),
        sub.nsmallest(3, "delta_net"),
        sub.nlargest(3,  "delta_local"),
        sub.nsmallest(3, "delta_local"),
    ]).drop_duplicates(subset="city")

    for _, row in extremes.iterrows():
        texts.append(ax.text(
            row["delta_net"], row["delta_local"],
            row["city_label"],
            fontsize=4.5, color=COLORS[region], zorder=5
        ))

# 零线
ax.axhline(0, color="grey", lw=0.5, ls="--", zorder=1)
ax.axvline(0, color="grey", lw=0.5, ls="--", zorder=1)

# 标签防重叠
adjust_text(texts, ax=ax,
            arrowprops=dict(arrowstyle="-", color="grey", lw=0.3))
# 过原点的等压力零线：delta_net + delta_local = 0
x_ref = np.array([x_lo, x_hi])
ax.plot(x_ref, -x_ref,
        color="grey", lw=0.6, ls=":",
        alpha=0.5, zorder=1,
        label="ΔTotal=0")
# 象限标注（取绘图后的实际范围）
x_lo, x_hi = ax.get_xlim()
y_lo, y_hi = ax.get_ylim()
quad_kw = dict(fontsize=6, color="#4a4a4a", alpha=0.6)
ax.text(x_lo * 0.98, y_hi * 0.98, "Net inflow↓\nLocal↑",  ha="left",  va="top",    **quad_kw)
ax.text(x_hi * 0.98, y_hi * 0.98, "Net inflow↑\nLocal↑",  ha="right", va="top",    **quad_kw)
ax.text(x_lo * 0.98, y_lo * 0.98, "Net inflow↓\nLocal↓",  ha="left",  va="bottom", **quad_kw)
ax.text(x_hi * 0.98, y_lo * 0.98, "Net inflow↑\nLocal↓",  ha="right", va="bottom", **quad_kw)

# ── 6. 合并图例（一行）────────────────────────────────────────────────────────

legend_handles = []

# Region
for region in ["East", "West"]:
    n = len(df_plot[df_plot["region"] == region])
    legend_handles.append(
        mlines.Line2D([], [],
                      color=COLORS[region],
                      marker=MARKERS[region],
                      linestyle="None",
                      markersize=5,
                      alpha=0.7,
                      label=f"{region} (n={n})")
    )

# 点大小档位
for val, label in [(5000, "5K"), (20000, "20K"), (50000, "50K")]:
    s = (val / size_vals.max()) * 60 + 8
    legend_handles.append(
        mlines.Line2D([], [],
                      color="grey",
                      marker="o",
                      linestyle="None",
                      markersize=np.sqrt(s),
                      alpha=0.45,
                      label=f"ΔTotal={label}")
    )

ax.legend(
    handles=legend_handles,
    ncol=5,
    fontsize=5,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    handletextpad=0.3,
    columnspacing=0.8,
)

# ── 7. 轴样式 ─────────────────────────────────────────────────────────────────

ax.set_xlabel("Change in net inflow patients (2020-2060)", fontsize=6)
ax.set_ylabel("Change in local patients (2020-2060)",      fontsize=6)
ax.xaxis.set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}K"))
ax.yaxis.set_major_formatter(
    matplotlib.ticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}K"))
ax.spines[["top","right"]].set_visible(False)
ax.spines[["bottom","left"]].set_linewidth(0.4)
ax.tick_params(width=0.4, length=2, labelsize=5)

# ── 8. 保存 ───────────────────────────────────────────────────────────────────

plt.tight_layout()
plt.subplots_adjust(bottom=0.22)

OUTFILE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTFILE, dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print(f"✓ Saved → {OUTFILE}")

/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_24485/1765333306.py:41: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cx = city_shp.geometry.centroid.x
/var/folders/xt/zxpvbvsx2zb75gd2395zm9000000gn/T/ipykernel_24485/1765333306.py:42: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cy = city_shp.geometry.centroid.y


总城市数: 165
       delta_net                                                       \
           count   mean     std      min    25%    50%    75%     max   
region                                                                  
East       143.0 -170.0  1887.0 -12361.0    2.0  216.0  464.0  2846.0   
West        22.0 -303.0  1169.0  -3895.0 -363.0   31.0  283.0  1052.0   

       delta_local          ...                 delta_total                  \
             count    mean  ...    75%      max       count    mean     std   
region                      ...                                               
East         143.0 -1684.0  ... -503.0  12393.0       143.0 -1854.0  3374.0   
West          22.0    13.0  ...  922.0   8187.0        22.0  -290.0  2426.0   

                                                 
            min     25%     50%    75%      max  
region                                           
East   -25947.0 -2922.0 -1153.0 -431.0  10818.0  
West    -3536.0 -1757.0  -52